In [ ]:
%load_ext autoreload
%autoreload 2

import plotly.express as px
import polars as pl

from deephit_cancer_comparison.constants import DATA_PATH

SEER_ORIGINAL_DATA_PATH = DATA_PATH / "seer_original" / "seer_dataset.txt"
CANCER_SPECIFIC_DATA_PATH = DATA_PATH / "cancer_specific_data"

In [ ]:
SEER_MISSING_PATTERNS = [
    "Unknown",
    "NA",
    "Blank(s)",
    "unknown",
    "blank(s)",
    "blank",
    "not applicable",
    "unknown/not applicable",
    "na",
    "999",
    "9999",
    "99999",
]
seer_df = pl.read_csv(
    SEER_ORIGINAL_DATA_PATH,
    null_values=SEER_MISSING_PATTERNS,
).drop(
    pl.col("Site recode ICD-O-3 2023 Revision")  # Using the expanded version of this variable
)

In [ ]:
col_map = {
    "Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)": "race_origin",
    "Age recode with single ages and 90+": "age",
    "Combined Summary Stage with Expanded Regional Codes (2004+)": "summary_stage",
    "Histology recode - broad groupings": "histology",
    "Marital status at diagnosis": "marital_status",
    "Sequence number": "sequence_number",
    "Site recode ICD-O-3 2023 Revision Expanded": "site_recode",
    "Tumor Size Over Time Recode (1988+)": "tumor_size",
    "SEER cause-specific death classification": "cause_specific_death",
    "SEER other cause of death classification": "other_cause_death",
    "Survival months": "survival_months",
    "Survival months flag": "survival_months_flag",
    "Vital status recode (study cutoff used)": "vital_status",
    "Grade Recode (thru 2017)": "grade",
    "Year of diagnosis": "year_dx",
    "Sex": "sex",
}
seer_df = seer_df.rename(col_map)

In [ ]:
top10cancers = (seer_df.group_by("site_recode").len().sort("len", descending=True).head(10))[
    "site_recode"
].to_list()

print("=== TOP 10 MOST PREVALENT CANCERS ===")
print("\n".join(f"{cancer}" for cancer in top10cancers))

seer_df = seer_df.filter(
    pl.col("site_recode").is_in(top10cancers)
)  # Considering only top 10 most prevalent cancers

In [ ]:
print("=== SHAPE ===")
print(f"Rows: {seer_df.shape[0]:,} | Columns: {seer_df.shape[1]:,}")

In [ ]:
print("=== DTYPES ===")
print("\n".join(f"{col} | {dtype}" for col, dtype in zip(seer_df.columns, seer_df.dtypes)))

In [ ]:
n_dupes = seer_df.is_duplicated().sum()
print("=== DUPLICATES ===")
print(f"Exact duplicate rows: {n_dupes:,}")

seer_df = seer_df.unique()  # Dropping exact duplicates
print(f"After dropping duplicates: {seer_df.shape[0]:,}")

In [ ]:
print("=== MISSINGNESS AUDIT ===")
miss_rows = []
for col in seer_df.columns:
    null_count = seer_df[col].null_count()
    pct = round(100 * null_count / seer_df.shape[0], 2)
    miss_rows.append(
        {
            "column": col,
            "null_count": null_count,
            "pct_missing": pct,
        }
    )

miss_df = pl.DataFrame(miss_rows).sort("pct_missing", descending=True)
miss_df

In [ ]:
print("=== SURVIVAL MONTHS SANITY CHECK ===")
temp_df = seer_df.with_columns(pl.col("survival_months").cast(pl.Float64, strict=False))
print(f"Negative values:    {(temp_df["survival_months"] < 0).sum()}")
print(f"Zero values:        {(temp_df["survival_months"] == 0).sum()}")
print(f"Values > 300 mo:    {(temp_df["survival_months"] > 300).sum()}")
with pl.Config(set_fmt_float="full"):
    print(temp_df["survival_months"].describe())

In [ ]:
cat_cols = [
    "sex",
    "race_origin",
    "summary_stage",
    "histology",
    "marital_status",
    "sequence_number",
    "grade",
    "cause_specific_death",
    "other_cause_death",
    "vital_status",
    "survival_months_flag",
]

print("=== CATEGORICAL VALUE COUNTS ===")
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(seer_df.group_by(col).agg(pl.len().alias("count")).sort("count", descending=True))

In [ ]:
print("=== YEAR OF DIAGNOSIS DISTRIBUTION ===")
(seer_df.group_by("year_dx").agg(pl.len().alias("count")).sort("year_dx", descending=True))

In [ ]:
print("=== TUMOR SIZE DISTRIBUTION ===")
temp_df = seer_df.with_columns(pl.col("tumor_size").cast(pl.Float64, strict=False))
with pl.Config(set_fmt_float="full"):
    print(temp_df["tumor_size"].describe())

In [ ]:
print("=== CANCER TYPE COUNTS ===")
(seer_df.group_by("site_recode").agg(pl.len().alias("count")).sort("count", descending=True))

In [ ]:
def count_seer_missing(series: pl.Series) -> int:
    return (
        series.cast(pl.Utf8).str.strip_chars().str.to_lowercase().is_in(SEER_MISSING_PATTERNS).sum()
    )


print("=== MISSINGNESS PER COHORT ===")

for col in seer_df.columns:
    print(f"\n--- {col} ---")
    rows = []
    for cancer in top10cancers:
        sub = seer_df.filter(pl.col("site_recode") == cancer)
        n = sub.shape[0]
        total_miss = sub[col].null_count() + count_seer_missing(sub[col])
        pct = round(100 * total_miss / n, 2)
        rows.append({"cancer": cancer, "n": n, "missing": total_miss, "pct_missing": pct})
    print(pl.DataFrame(rows))

In [ ]:
seer_clean_df = (
    seer_df.drop(pl.col("grade"))  # Dropped due to really high missingness ratio
    .filter(
        (pl.col("survival_months").is_not_null())  # Drop rows where survival months are not known
        & (pl.col("sequence_number") == "One primary only")  # Only patients with first cancers
        & (pl.col("year_dx").ge(2004))  # Drop all patients diagnosed before 2004
    )
    .with_columns(
        pl.col("summary_stage").fill_null("Unknown"),  # Encode nulls as "Unknown"
        pl.col("marital_status").fill_null("Unknown"),  # Encode nulls as "Unknown"
    )
    .drop("sequence_number")  # Drop after filtering on
)

In [ ]:
print("=== RATIO OF DEATHS IN FIRST MONTH ===")
(
    seer_clean_df.group_by("site_recode")
    .agg(
        (pl.col("survival_months") == 0).sum().alias("zero_survival_months"),
        (pl.col("survival_months") != 0).sum().alias("not_zero_survival_months"),
    )
    .with_columns(
        (pl.col("zero_survival_months") / pl.col("not_zero_survival_months") * 100).alias(
            "pct_zero"
        )
    )
    .sort("zero_survival_months", descending=True)
)

# TODO
- Vital status column
- Survival months flag column

In [ ]:
print("=== FINAL DATASET SUMMARY ===")
print(f"Shape: {seer_clean_df.shape[0]:,} rows x {seer_clean_df.shape[1]} columns")
print(f"\nColumns: {seer_clean_df.columns}")

print("\nRemaining nulls per column:")
for col in seer_clean_df.columns:
    n = seer_clean_df[col].null_count()

    print(f"  {col}: {n:,}")

print("\nCohort sizes:")
print(seer_clean_df.group_by("site_recode").agg(pl.len().alias("n")).sort("n", descending=True))

In [ ]:
cohort_df = (
    seer_clean_df.with_columns(
        pl.when(
            (pl.col("cause_specific_death") == "Dead (attributable to this cancer dx)")
            & (pl.col("vital_status") == "Dead")
        )
        .then(0)
        .when(
            (pl.col("cause_specific_death") != "Dead (attributable to this cancer dx)")
            & (pl.col("vital_status") == "Dead")
        )
        .then(1)
        .otherwise(2)
        .alias("outcome")
    )
    .group_by("site_recode")
    .agg(
        [
            pl.len().alias("cohort_size"),
            (pl.col("outcome") == 0).sum().alias("n_cancer_deaths"),
            (pl.col("outcome") == 1).sum().alias("n_other_deaths"),
            (pl.col("outcome") == 2).sum().alias("n_censored"),
        ]
    )
    .sort("cohort_size", descending=True)
)
cohort_df

In [ ]:
seer_clean_df

In [ ]:
seer_clean_df["age"].value_counts()

In [ ]:
YEAR_MIN = seer_clean_df["year_dx"].min()
YEAR_MAX = seer_clean_df["year_dx"].max()

seer_encoded = seer_clean_df.with_columns(
    # Binary coding of SEX variable
    pl.col("sex").replace({"Male": 0, "Female": 1}).cast(pl.Int8).alias("sex"),
    # Min-max normalization for YEAR_DX -> [0, 1]
    ((pl.col("year_dx") - YEAR_MIN) / (YEAR_MAX - YEAR_MIN)).alias("year_dx"),
    # Min-max normalization for AGE -> [0, 1]
    (int(pl.col("age")[:2]).alias("age")),
)

# One-Hot encoding for RACE_ORIGIN (DeepHit 2018)
RACE_COL = "race_origin"

RACE_CATEGORIES = seer_clean_df[RACE_COL].unique().to_list()
RACE_CATEGORIES.remove("Non-Hispanic White")

RACE_DUMMIES_MAPPING = {
    f"{RACE_COL}_Hispanic (All Races)": "race_hispanic",
    f"{RACE_COL}_Non-Hispanic American Indian/Alaska Native": "race_american_indian_alaska_native",
    f"{RACE_COL}_Non-Hispanic Asian or Pacific Islander": "race_asian_pacific_islander",
    f"{RACE_COL}_Non-Hispanic Black": "race_black",
    f"{RACE_COL}_Non-Hispanic Unknown Race": "race_unknown",
}

race_dummies = (seer_clean_df.select(pl.col(RACE_COL)).to_dummies(columns=[RACE_COL])).drop(
    f"{RACE_COL}_Non-Hispanic White"
)

race_dummies = race_dummies.with_columns([pl.col(c).cast(pl.Int8) for c in race_dummies.columns])

seer_encoded = pl.concat([seer_encoded.drop(RACE_COL), race_dummies], how="horizontal").rename(
    RACE_DUMMIES_MAPPING
)

seer_encoded

# GRAPHS

In [ ]:
cohort_long = (
    cohort_df.select(
        ["site_recode", "n_cancer_deaths", "n_other_deaths", "n_censored", "cohort_size"]
    )
    .unpivot(
        on=["n_cancer_deaths", "n_other_deaths", "n_censored"],
        index=["site_recode", "cohort_size"],
        variable_name="outcome_type",
        value_name="count",
    )
    .with_columns(
        pl.col("outcome_type").replace(
            {
                "n_cancer_deaths": "Cancer-specific death",
                "n_other_deaths": "Other-cause death",
                "n_censored": "Censored / Alive",
            }
        )
    )
    .sort("cohort_size", descending=True)
)

fig = px.bar(
    cohort_long.to_pandas(),
    x="site_recode",
    y="count",
    color="outcome_type",
    color_discrete_map={
        "Cancer-specific death": "#27187e",
        "Other-cause death": "#758bfd",
        "Censored / Alive": "#aeb8fe",
    },
    title="Cohort Sizes and Outcome Distribution - Top 10 Cancer Types (SEER, 2004–2021)",
    labels={"count": "Number of patients", "site_recode": "", "outcome_type": "Outcome"},
    height=600,
    category_orders={"site_recode": cohort_df["site_recode"].to_list()},
)

fig.update_layout(
    barmode="stack",
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial, sans-serif", color="#1a1a1a", size=13),
    title=dict(
        font=dict(
            size=25,
            color="#1a1a1a",
        ),
        xanchor="center",
        x=0.5,
    ),
    legend=dict(
        orientation="v",
        x=0.98,
        y=0.98,
        xanchor="right",
        yanchor="top",
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
    yaxis=dict(
        gridcolor="#eeeeee",
        title=dict(
            text="Number of Patients",
            font=dict(size=20),
        ),
    ),
    xaxis=dict(tickangle=-30),
    margin=dict(l=60, r=30, t=60, b=130),
)

fig.show()

fig.write_image("cohort_plot.png", width=1200, height=600, scale=2)

In [ ]:
import plotly.graph_objects as go

cohorts = [
    "Pancreas",
    "Lung & Bronchus",
    "Colon & Rectum",
    "NHL",
    "Kidney",
    "Breast",
    "Corpus Uteri",
    "Melanoma",
    "Prostate",
    "Thyroid",
]

# Replace these with your actual bootstrapped values
means = [0.782, 0.761, 0.734, 0.718, 0.703, 0.689, 0.671, 0.658, 0.634, 0.601]
lower = [0.769, 0.748, 0.718, 0.700, 0.688, 0.672, 0.651, 0.636, 0.610, 0.574]
upper = [0.795, 0.774, 0.750, 0.736, 0.718, 0.706, 0.691, 0.680, 0.658, 0.628]

# Sort ascending so highest performer appears at top of horizontal chart
order = sorted(range(len(means)), key=lambda i: means[i])
cohorts = [cohorts[i] for i in order]
means = [means[i] for i in order]
lower = [lower[i] for i in order]
upper = [upper[i] for i in order]

error_minus = [means[i] - lower[i] for i in range(len(means))]
error_plus = [upper[i] - means[i] for i in range(len(means))]

fig = go.Figure()

# CI bars (wide, light blue)
fig.add_trace(
    go.Bar(
        y=cohorts,
        x=[upper[i] - lower[i] for i in range(len(means))],
        base=lower,
        orientation="h",
        marker=dict(color="#B5D4F4", line=dict(color="#85B7EB", width=0.5)),
        width=0.5,
        name="95% CI (bootstrapped)",
        hovertemplate="<b>%{y}</b><br>95% CI: [%{base:.3f}, %{x:.3f}]<extra></extra>",
        customdata=upper,
    )
)

# Mean markers (narrow, dark blue)
fig.add_trace(
    go.Bar(
        y=cohorts,
        x=[0.004] * len(means),
        base=[m - 0.002 for m in means],
        orientation="h",
        marker=dict(color="#185FA5"),
        width=0.5,
        name="Ctd-index (mean)",
        hovertemplate="<b>%{y}</b><br>Ctd-index: %{customdata:.3f}<extra></extra>",
        customdata=means,
    )
)

fig.update_layout(
    barmode="overlay",
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=420,
    margin=dict(l=140, r=40, t=50, b=60),
    title=dict(
        text="Bootstrapped 95% Confidence Intervals - Ctd-index per Cancer Cohort",
        font=dict(size=17, color="#1a1a1a"),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(
        title=dict(text="Time-dependent concordance index (Ctd-index)", font=dict(size=15)),
        range=[0.54, 0.84],
        tickformat=".2f",
        gridcolor="#eeeeee",
        tickfont=dict(size=11),
    ),
    yaxis=dict(
        tickfont=dict(size=12),
        gridcolor="white",
    ),
    legend=dict(
        orientation="v",
        y=0.02,
        x=0.7,
        font=dict(size=11),
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
)

fig.show()
fig.write_image("ctd_index_ci.png", width=900, height=420, scale=2)

In [ ]:
import plotly.graph_objects as go

variables = [
    "sex",
    "year_dx",
    "race_origin",
    "age",
    "summary_stage",
    "histology",
    "marital_status",
    "tumor_size",
]

# Replace with actual aggregated SurvSHAP(t) values per cohort
importance_pancreas = [0.021, 0.018, 0.024, 0.198, 0.312, 0.087, 0.031, 0.243]
importance_thyroid = [0.034, 0.041, 0.028, 0.089, 0.401, 0.223, 0.055, 0.067]

# Sort by average importance across both cancers for consistent ordering
avg = [(importance_pancreas[i] + importance_thyroid[i]) / 2 for i in range(len(variables))]
order = sorted(range(len(variables)), key=lambda i: avg[i])

variables = [variables[i] for i in order]
importance_pancreas = [importance_pancreas[i] for i in order]
importance_thyroid = [importance_thyroid[i] for i in order]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=variables,
        x=importance_pancreas,
        orientation="h",
        name="Pancreas",
        marker=dict(color="#185FA5", line=dict(width=0)),
        width=0.35,
        hovertemplate="<b>%{y}</b><br>Pancreas: %{x:.3f}<extra></extra>",
    )
)


fig.add_trace(
    go.Bar(
        y=variables,
        x=importance_thyroid,
        orientation="h",
        name="Thyroid",
        marker=dict(color="#758bfd", line=dict(width=0)),
        width=0.35,
        hovertemplate="<b>%{y}</b><br>Thyroid: %{x:.3f}<extra></extra>",
    )
)


fig.update_layout(
    barmode="group",
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=420,
    margin=dict(l=120, r=40, t=40, b=60),
    title=dict(
        text="Feature Importances - SurvSHAP(t) Aggregated per Cohort",
        font=dict(size=18, color="#1a1a1a"),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(
        title=dict(text="Mean absolute SurvSHAP(t) value", font=dict(size=13)),
        gridcolor="#eeeeee",
        tickfont=dict(size=11),
    ),
    yaxis=dict(tickfont=dict(size=12), gridcolor="white", ticklabelstandoff=10),
    legend=dict(
        orientation="v",
        y=0.02,
        x=0.85,
        font=dict(size=11),
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
)

fig.show()
fig.write_image("feature_importances.png", width=900, height=420, scale=2)